# State Space Models (Mamba)

দুটি demo:

1. S4-এর মূল duality: একটি fixed-A/B/C linear state-space recurrence **দুইভাবে** গণনা করা হয় — ক্রমিকভাবে (একবারে এক ধাপ) এবং একটি নির্দিষ্ট kernel-এর বিপরীতে একটি একক global convolution হিসেবে — এবং যাচাই করা হয় যে দুটো floating-point precision পর্যন্ত মিলে যায়।
2. একটি selective (Mamba-style, input-নির্ভর gating) SSM বনাম একটি fixed (S4-style) SSM-এর তুলনা: sequence-এর দৈর্ঘ্য বাড়ার সাথে সাথে **শেষ** state এখনও **প্রথম** input-এর উপর কতটা নির্ভর করে — ঠিক একই পদ্ধতি যা Phase 01 Lesson 3-এ একটি vanilla RNN-কে LSTM-এর সাথে তুলনা করতে ব্যবহৃত হয়েছিল।

**চালানোর নিয়ম:** উপর থেকে নিচ পর্যন্ত সব cell ক্রমানুসারে চালান। শুধু NumPy লাগে, CPU-তে কয়েক সেকেন্ডেই চলে।

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

## 1. Sequential recurrence বনাম global convolution — S4 duality

এখানে একই linear recurrence `h_t = A h_{t-1} + B x_t ; y_t = C h_t` দুইভাবে গণনা করা হয়: `sequential_ssm` প্রতি ধাপে একবার করে loop চালায়, আর `convolution_ssm` প্রথমে kernel `K_i = C @ A^i @ B` তৈরি করে তারপর একটি causal convolution চালায়। Demo-টি দুটি output-এর সর্বোচ্চ পার্থক্য দেখায় — যা প্রায় শূন্য হওয়া উচিত।

In [ ]:
# ---------------------------------------------------------------------------
# 1. Sequential recurrence বনাম global convolution -- S4 duality
# ---------------------------------------------------------------------------

def sequential_ssm(x_seq, A, B, C):
    """h_t = A h_{t-1} + B x_t ; y_t = C h_t, একবারে এক ধাপ করে চালানো হয়।"""
    N = A.shape[0]
    h = np.zeros(N)
    ys = []
    for x_t in x_seq:
        h = A @ h + B * x_t
        ys.append((C @ h))
    return np.array(ys)


def convolution_ssm(x_seq, A, B, C):
    """হুবহু একই recurrence, kernel K_i = C @ A^i @ B-এর বিপরীতে একটি একক
    causal convolution হিসেবে গণনা করা (README section 3 দেখুন)।"""
    T = len(x_seq)
    N = A.shape[0]
    kernel = np.zeros(T)
    A_power = np.eye(N)
    for i in range(T):
        kernel[i] = C @ (A_power @ B)
        A_power = A @ A_power

    ys = np.zeros(T)
    for t in range(T):
        # y_t = sum_{k=0}^{t} kernel[t-k] * x_k   (0-indexed causal convolution)
        ys[t] = np.dot(kernel[: t + 1][::-1], x_seq[: t + 1])
    return ys


def duality_demo():
    print("=" * 70)
    print("1. RECURRENCE == CONVOLUTION: THE CORE S4 DUALITY")
    print("=" * 70)

    N = 4  # state dimension
    T = 20
    # একটি fixed, stable system: eigenvalue-গুলো unit circle-এর যথেষ্ট ভেতরে,
    # যাতে T=20 ধাপে recurrence বিস্ফোরিত (blow up) না হয়।
    raw = rng.normal(scale=0.5, size=(N, N))
    A = raw / (np.max(np.abs(np.linalg.eigvals(raw))) * 1.5)
    B = rng.normal(size=N)
    C = rng.normal(size=N)

    x_seq = rng.normal(size=T)

    y_sequential = sequential_ssm(x_seq, A, B, C)
    y_convolution = convolution_ssm(x_seq, A, B, C)

    max_diff = np.abs(y_sequential - y_convolution).max()
    print(f"State dimension N={N}, sequence length T={T}")
    print(f"Max abs difference between sequential and convolution outputs: {max_diff:.2e}")
    print("\nFirst 5 outputs, sequential vs. convolution:")
    for t in range(5):
        print(f"  t={t}  sequential={y_sequential[t]:.6f}   convolution={y_convolution[t]:.6f}")

    print("\n-> Identical (to floating-point precision) despite being computed two")
    print("   completely different ways: one step-by-step with a hard dependency")
    print("   chain (O(T), sequential-only), the other as one shot against a fixed")
    print("   kernel (parallelizable across the whole sequence at once). This")
    print("   agreement IS the duality that makes S4-style models trainable in")
    print("   parallel like attention, yet able to run as an O(1)-per-step")
    print("   recurrence like an RNN at inference time.")


duality_demo()

## 2. Selective (Mamba-style) বনাম fixed (S4-style) SSM: memory retention

Fixed SSM প্রতি ধাপে একই decay factor ব্যবহার করে, input-এর বিষয়বস্তু যা-ই হোক না কেন। Selective SSM-এ retention factor `a_t` প্রতিটি ধাপে বর্তমান input থেকে গণনা করা হয় — ঠিক LSTM-এর forget gate-এর মতো, এবং একটি বড় positive bias দিয়ে যাতে default আচরণ হয় "দৃঢ়ভাবে ধরে রাখা"। Demo-টি central-difference gradient দিয়ে মাপে শেষ state প্রথম input-এর উপর কতটা নির্ভর করে, বিভিন্ন sequence length-এর জন্য।

In [ ]:
# ---------------------------------------------------------------------------
# 2. Selective (Mamba-style) বনাম fixed (S4-style) SSM: memory retention
# ---------------------------------------------------------------------------

def fixed_ssm_forward(x_seq, decay, hidden_dim):
    """একটি fixed, non-selective SSM: প্রতি ধাপে একই decay factor, input-এর
    বিষয়বস্তু যা-ই হোক -- একটি সাধারণ exponential-decay memory।"""
    h = np.zeros(hidden_dim)
    for x_t in x_seq:
        h = decay * h + (1 - decay) * x_t
    return h


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def selective_ssm_forward(x_seq, W_delta, b_delta, hidden_dim):
    """একটি selective SSM: retention factor a_t = sigmoid(W_delta . x_t + b_delta)
    প্রতিটি ধাপে বর্তমান INPUT থেকে গণনা করা হয়, ঠিক LSTM-এর forget gate-এর
    মতো (Phase 01 Lesson 3)। b_delta-কে বড় ও positive মান দিয়ে initialize করা
    হয় -- Phase 01 Lesson 3-এর সেই একই forget-bias কৌশল -- যাতে default আচরণ
    হয় "দৃঢ়ভাবে ধরে রাখা," অথচ কোনো নির্দিষ্ট input-এর বিষয়বস্তু প্রয়োজন হলে
    সেটি (W_delta-এর মাধ্যমে) a_t-কে নামিয়ে দিয়ে state-কে বেছে বেছে overwrite
    করতে পারে।"""
    h = np.zeros(hidden_dim)
    for x_t in x_seq:
        a_t = sigmoid(W_delta * x_t + b_delta)   # elementwise, input-নির্ভর retention
        h = a_t * h + (1 - a_t) * x_t
    return h


def gradient_of_output_wrt_first_input(forward_fn, x_seq, eps=1e-4):
    """x_seq[0]-এর সাপেক্ষে sum(h_final)-এর central-difference gradient --
    Phase 01 Lesson 3-এর example.py-তে vanilla RNN-কে LSTM-এর সাথে তুলনা করতে
    ব্যবহৃত হুবহু একই পদ্ধতি।"""
    x0 = x_seq[0]
    seq_plus = x_seq.copy()
    seq_minus = x_seq.copy()
    seq_plus[0] = x0 + eps
    seq_minus[0] = x0 - eps
    out_plus = forward_fn(seq_plus).sum()
    out_minus = forward_fn(seq_minus).sum()
    return (out_plus - out_minus) / (2 * eps)


def selectivity_demo():
    print("\n" + "=" * 70)
    print("2. SELECTIVE (MAMBA-STYLE) vs. FIXED (S4-STYLE) SSM: MEMORY RETENTION")
    print("=" * 70)
    print("Measuring: how much does the FINAL state still depend on the FIRST")
    print("input, as sequence length grows? (Same question, same method, as")
    print("Phase 01 Lesson 3's vanilla-RNN-vs-LSTM gradient comparison.)\n")

    hidden_dim = 8
    FIXED_DECAY = 0.85   # প্রতি ধাপে একটি মাঝারি, বিষয়বস্তু-নিরপেক্ষ decay

    W_delta = rng.normal(scale=0.3, size=hidden_dim)
    b_delta = np.full(hidden_dim, 3.0)   # LSTM-style forget-bias কৌশল: default-এ ধরে রাখা

    max_len = 60
    full_sequence = rng.normal(size=max_len)

    print(f"{'seq_len':>8}  {'fixed SSM grad':>16}  {'selective SSM grad':>20}")
    for T in [2, 5, 10, 20, 40, 60]:
        x_seq = full_sequence[:T]

        fixed_grad = gradient_of_output_wrt_first_input(
            lambda seq: fixed_ssm_forward(seq, FIXED_DECAY, hidden_dim), x_seq
        )
        selective_grad = gradient_of_output_wrt_first_input(
            lambda seq: selective_ssm_forward(seq, W_delta, b_delta, hidden_dim), x_seq
        )
        print(f"{T:>8}  {abs(fixed_grad):>16.8f}  {abs(selective_grad):>20.8f}")

    print("\n-> The fixed SSM's dependence on the first input decays geometrically")
    print("   (multiplying by the same 0.85 factor every step, exactly like a")
    print("   vanilla RNN's repeated tanh-derivative multiplication) -- it has no")
    print("   way to decide, per token, to protect that early signal. The")
    print("   selective SSM's retention gate defaults toward 'keep the existing")
    print("   state' (via the large positive bias, the same trick used to")
    print("   initialize an LSTM's forget gate), so its dependence on the first")
    print("   input decays far more slowly across the same range of sequence")
    print("   lengths -- input-dependent gating, not a bigger state, is what")
    print("   buys the extra memory.")


selectivity_demo()

## সবগুলো demo একসাথে

`main()` function-টি উপরের সব demo একবারে ক্রমানুসারে চালায়। উপরের cell-গুলোতে demo-গুলো ইতিমধ্যে চলেছে, তাই দুবার চলা এড়াতে call-টি comment করে রাখা হয়েছে।

In [ ]:
def main():
    duality_demo()
    selectivity_demo()


# main()  # সব demo আবার একসাথে চালাতে uncomment করুন